# Scalability Benchmark

Measures runtime scaling with data dimensions.

- **Runtime vs rows**: Fixed N_cols=10, vary rows from 50 to 2000
- **Runtime vs columns**: Fixed N_rows=200, vary columns from 5 to 200
- **Per-sweep amortization**: lax.scan benefit as n_sweeps increases
- **JIT compilation time**: Overhead vs problem size

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    # Checkout branch: try local first, then create from remote tracking branch
    checkout = subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR)
    if checkout.returncode != 0:
        subprocess.run(
            ["git", "checkout", "-b", BRANCH, f"origin/{BRANCH}"], cwd=WORKDIR, check=True
        )
    else:
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import json
import time

import jax
import jax.numpy as jnp
import numpy as np

from benchmarks.utils import create_results_dir, detect_platform, make_benchmark_data
from crosscat import initialize, pack_state, packed_gibbs_sweep

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Runtime vs Number of Rows

Fixed N_cols=10, vary rows from 50 to 2000.

In [ ]:
def time_sweep(key, data, col_types, n_sweeps=5, n_warmup=1):
    """Time packed_gibbs_sweep: (compile_time, per_sweep_time)."""
    k1, k2, k3 = jax.random.split(key, 3)
    state = initialize(k1, data, col_types).state
    packed = pack_state(state)
    t0 = time.perf_counter()
    packed_w = packed_gibbs_sweep(k2, packed, data, n_sweeps=n_warmup)
    packed_w.column_assignments.block_until_ready()
    compile_time = time.perf_counter() - t0
    t0 = time.perf_counter()
    packed_out = packed_gibbs_sweep(k3, packed_w, data, n_sweeps=n_sweeps)
    packed_out.column_assignments.block_until_ready()
    total_time = time.perf_counter() - t0
    return compile_time, total_time / n_sweeps


base_key = jax.random.key(42)
row_counts = [50, 100, 200, 500, 1000, 2000]
n_cols = 10
rows_results = []

print("=== Scalability vs N_rows (N_cols=10) ===")
for n_rows in row_counts:
    key = jax.random.fold_in(base_key, n_rows)
    data, col_types = make_benchmark_data(key, n_rows, n_cols)
    k_time = jax.random.fold_in(key, 999)
    compile_t, sweep_t = time_sweep(k_time, data, col_types, n_sweeps=5)
    print(f"  N_rows={n_rows:5d}: compile={compile_t:.2f}s, sweep={sweep_t:.4f}s")
    rows_results.append(
        {
            "n_rows": n_rows,
            "n_cols": n_cols,
            "compile_time": compile_t,
            "per_sweep_time": sweep_t,
        }
    )

## 3. Runtime vs Number of Columns

Fixed N_rows=200, vary columns from 5 to 200.

In [ ]:
col_counts = [5, 10, 20, 50, 100, 200]
n_rows = 200
cols_results = []

print("=== Scalability vs N_cols (N_rows=200) ===")
for n_cols in col_counts:
    key = jax.random.fold_in(base_key, n_cols + 10000)
    data, col_types = make_benchmark_data(key, n_rows, n_cols)
    k_time = jax.random.fold_in(key, 999)
    compile_t, sweep_t = time_sweep(k_time, data, col_types, n_sweeps=5)
    print(f"  N_cols={n_cols:5d}: compile={compile_t:.2f}s, sweep={sweep_t:.4f}s")
    cols_results.append(
        {
            "n_rows": n_rows,
            "n_cols": n_cols,
            "compile_time": compile_t,
            "per_sweep_time": sweep_t,
        }
    )

## 4. Per-Sweep Amortization

lax.scan amortization: per-sweep time drops as n_sweeps increases.

In [ ]:
sweep_counts = [1, 5, 10, 50, 100]
n_rows, n_cols = 200, 10
sweeps_results = []

print("=== Per-sweep time vs N_sweeps (200x10) ===")
key = jax.random.fold_in(base_key, 77777)
data, col_types = make_benchmark_data(key, n_rows, n_cols)
k_init = jax.random.fold_in(key, 0)
state = initialize(k_init, data, col_types).state
packed = pack_state(state)
k_warmup = jax.random.fold_in(key, 1)
packed = packed_gibbs_sweep(k_warmup, packed, data, n_sweeps=1)
packed.column_assignments.block_until_ready()

for n_sw in sweep_counts:
    k_run = jax.random.fold_in(key, n_sw)
    t0 = time.perf_counter()
    out = packed_gibbs_sweep(k_run, packed, data, n_sweeps=n_sw)
    out.column_assignments.block_until_ready()
    total = time.perf_counter() - t0
    per_sweep = total / n_sw
    print(f"  N_sweeps={n_sw:5d}: total={total:.3f}s, per_sweep={per_sweep:.4f}s")
    sweeps_results.append({"n_sweeps": n_sw, "total_time": total, "per_sweep_time": per_sweep})

## 5. Scalability Plots

In [ ]:
import matplotlib.pyplot as plt

results_dir = create_results_dir("scalability")
all_results = {
    "vs_rows": rows_results,
    "vs_cols": cols_results,
    "vs_sweeps": sweeps_results,
    "backend": platform["backend"],
    "device": str(jax.devices()[0]),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (a) rows
ax = axes[0]
xs = [r["n_rows"] for r in rows_results]
ys = [r["per_sweep_time"] for r in rows_results]
ax.plot(xs, ys, "o-", color="#2196F3", linewidth=2, markersize=6)
ax.set_xlabel("Number of rows")
ax.set_ylabel("Time per sweep (s)")
ax.set_title("(a) Scaling with rows")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (b) cols
ax = axes[1]
xs = [r["n_cols"] for r in cols_results]
ys = [r["per_sweep_time"] for r in cols_results]
ax.plot(xs, ys, "s-", color="#4CAF50", linewidth=2, markersize=6)
ax.set_xlabel("Number of columns")
ax.set_ylabel("Time per sweep (s)")
ax.set_title("(b) Scaling with columns")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (c) compile
ax = axes[2]
sizes_r = [r["n_rows"] * r["n_cols"] for r in rows_results]
compile_r = [r["compile_time"] for r in rows_results]
sizes_c = [r["n_rows"] * r["n_cols"] for r in cols_results]
compile_c = [r["compile_time"] for r in cols_results]
ax.scatter(sizes_r, compile_r, marker="o", color="#2196F3", label="Vary rows", s=40)
ax.scatter(sizes_c, compile_c, marker="s", color="#4CAF50", label="Vary cols", s=40)
ax.set_xlabel("Table size (rows x cols)")
ax.set_ylabel("JIT compile time (s)")
ax.set_title("(c) Compilation overhead")
ax.set_xscale("log")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(results_dir / "scalability.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Save Results

In [ ]:
import shutil


# Save JSON
def convert(obj):
    """Convert numpy/jax types for JSON serialization."""
    if isinstance(obj, (np.integer, jnp.integer)):
        return int(obj)
    if isinstance(obj, (np.floating, jnp.floating, float)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


with open(results_dir / "scalability_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=convert)

print(f"Results saved to {results_dir}")

# Archive
archive = Path("benchmarks/results/scalability_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")